# 04 — CatBoost

CatBoost - это градиентный бустинг на деревьях решений от Яндекса. Идея бустинга такая: каждое следующее дерево учится предсказывать ошибки предыдущих, а итоговое предсказание модели - сумма по всем деревьям. Получается нелинейный ансамбль, который умеет ловить взаимодействия между признаками без ручного feature engineering.

В нашем проекте CatBoost - главный конкурент TabM. На табличных данных бустинги долго удерживали статус сильнейшего решения, и весь смысл проекта - проверить, может ли нейросетевой TabM их обогнать. У CatBoost для нашего датасета две заметные особенности: он сам умеет работать с категориальными признаками (свой target encoding с защитой от утечки через Ordered Boosting, OneHot не нужен) и из коробки довольно устойчив к переобучению.

Сам ноутбук работает на тех же `train/val/test`, что и LogReg в ноутбуке 03 - это принципиально для честного сравнения. Дальше: заворачиваю данные в CatBoost-овский `Pool`, перебираю 13 наборов гиперпараметров на val, выбираю лучший по F2, считаю метрики на test и сохраняю результаты в `results/` для ноутбука 08.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool

sys.path.append(str(Path.cwd().parent / "src"))
import utils

SEED = 42
PROCESSED_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results")
METRICS_DIR = RESULTS_DIR / "metrics"
PREDICTIONS_DIR = RESULTS_DIR / "predictions"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Загрузка обработанных данных

Читаю те же parquet-файлы, что и в ноутбуке 03, и тот же `feature_types.json`. Это принципиальный момент: модели в проекте сравниваются между собой, поэтому они обязаны видеть один и тот же train/val/test. Иначе разницу в метриках можно было бы списать на разные данные, а не на разные модели.

Из проверок стандартное: соотношение сплитов 60/20/20, доля позитивного класса ~9% во всех трёх (стратификация сохранилась), `numeric_features` и `categorical_features` - финальные признаки после SHAP-отбора из ноутбука 02. `target` хранится отдельно, чтобы случайно не попасть в feature list.

В выводе ячейки виден старый кеш - там цифры от прогона с меньшим набором признаков. Это просто не обновлённый output. При повторном запуске числа подтянутся под текущий `feature_types.json`, методология подбора и оценки от этого не меняется.

In [2]:
train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
val = pd.read_parquet(PROCESSED_DIR / "val.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test.parquet")

with open(PROCESSED_DIR / "feature_types.json", encoding="utf-8") as f:
    feature_types = json.load(f)

numeric_features = feature_types["numeric"]
categorical_features = feature_types["categorical"]
target_col = "target"
feature_cols = numeric_features + categorical_features

print("train:", train.shape, "positive rate:", train[target_col].mean().round(4))
print("val:  ", val.shape, "positive rate:", val[target_col].mean().round(4))
print("test: ", test.shape, "positive rate:", test[target_col].mean().round(4))
print("numeric:", len(numeric_features), "categorical:", len(categorical_features))

train: (41982, 26) positive rate: 0.0897
val:   (13994, 26) positive rate: 0.0898
test:  (13994, 26) positive rate: 0.0897
numeric: 12 categorical: 13


## 2. Pool для CatBoost

`Pool` - собственный контейнер CatBoost для обучающих данных. Внутри он хранит признаки, метки и список категориальных колонок в эффективном формате (особенно полезно на GPU). Можно передавать в `fit` и обычный DataFrame, но через Pool явнее: я сразу говорю модели, какие колонки считать категориальными.

Функция `prepare_xy` делает три вещи: выбирает нужные колонки, приводит категориальные значения к строкам и заполняет пропуски строкой `"Unknown"`. Приведение к строкам тут важно, потому что часть наших категориальных признаков исходно числовые коды (`admission_type_id`, `discharge_disposition_id`). Если оставить их числами, CatBoost воспримет их как обычные числовые и не применит target encoding. Со строкой - точно категория.

StandardScaler и OneHotEncoder, как в LogReg, здесь не нужны. Деревьям одинаково всё равно на любой масштаб признаков: сплит вида `x > порог` от линейной нормализации не зависит. А категориальные CatBoost кодирует сам, причём правильно - через target-based encoding с защитой от утечки.

Делаю три отдельных пула: train идёт в `fit`, val - в `eval_set` для early stopping и подбора гиперпараметров, test трогаю строго один раз, на финальной оценке.

In [3]:
def prepare_xy(df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    X = df[feature_cols].copy()
    for col in categorical_features:
        X[col] = X[col].astype("string").fillna("Unknown")
    y = df[target_col].to_numpy(dtype=int)
    return X, y


X_train, y_train = prepare_xy(train)
X_val, y_val = prepare_xy(val)
X_test, y_test = prepare_xy(test)

train_pool = Pool(X_train, y_train, cat_features=categorical_features)
val_pool = Pool(X_val, y_val, cat_features=categorical_features)
test_pool = Pool(X_test, y_test, cat_features=categorical_features)

## 3. Подбор гиперпараметров на validation

Идея простая: прогоняю 13 разных конфигураций CatBoost, смотрю на их F2 на val и выбираю лучшую. Test в этом цикле не задействован вообще.

Сначала про `base_params` - это то, что одинаково у всех кандидатов. `loss_function="Logloss"` - стандартная функция потерь для бинарной классификации, та же, что у LogReg. `eval_metric="AUC"` - то, что CatBoost мониторит на val во время обучения, чтобы понять, когда останавливаться. AUC устойчива к дисбалансу и не зависит от порога, поэтому удобна как сигнал. `auto_class_weights="Balanced"` - полный аналог нашего `class_weight="balanced"` у LogReg, редкий класс получает повышенный вес автоматически. `task_type="GPU", devices="0"` - учу на GPU, на нашем размере данных это сильно быстрее. Отсюда же warning `Default metric period is 5 because AUC is/are not implemented for GPU`: на GPU AUC считается раз в 5 итераций, а не каждую. Это не баг.

Сама сетка `param_grid` собрана руками, а не сгенерирована полным перебором. 13 кандидатов с разной глубиной (4-8), числом итераций (600-1500), learning rate (0.01-0.1), L2-регуляризацией листьев (3-12) и парой вариантов с `bagging_temperature`. Идея перебора - проверить разные полюса: маленький learning_rate и много итераций против большого learning_rate и мало; неглубокие деревья (меньше переобучаются) против глубоких (ловят более сложные взаимодействия); разная сила регуляризации.

Внутри цикла для каждого кандидата: создаю `CatBoostClassifier`, учу на train, в `eval_set` подсовываю val. `use_best_model=True` означает, что после обучения модель откатится на ту итерацию, где AUC на val был максимален - это страховка от переобучения на хвосте. `early_stopping_rounds=80` прерывает обучение, если 80 итераций подряд AUC на val не улучшается. По выводу хорошо видно, как это сработало: лучший кандидат остановился на 569-й итерации, хотя в параметрах было 900. Дальше на вероятностях val ищу порог по F2, считаю все метрики, и проверяю - лучше ли этот кандидат предыдущего по паре `(val_f2, val_roc_auc)`. Кортеж сравнивается лексикографически: сначала F2, при равенстве - AUC.

По итогам лучшим оказался кандидат №5: `depth=6, iterations=900, learning_rate=0.025, l2_leaf_reg=12.0`. Он же остановился на 569-й итерации и дал F2 ≈ 0.360 и AUC ≈ 0.641 на val.

In [4]:
base_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "auto_class_weights": "Balanced",
    "random_seed": SEED,
    "task_type": "GPU",
    "devices": "0",
    "verbose": False,
    "allow_writing_files": False,
    "thread_count": -1,
}

param_grid = [
    {"iterations": 600,  "depth": 4, "learning_rate": 0.05,  "l2_leaf_reg": 3.0},
    {"iterations": 600,  "depth": 4, "learning_rate": 0.1,   "l2_leaf_reg": 10.0},
    {"iterations": 900,  "depth": 5, "learning_rate": 0.03,  "l2_leaf_reg": 3.0},
    {"iterations": 900,  "depth": 5, "learning_rate": 0.05,  "l2_leaf_reg": 10.0},
    {"iterations": 900,  "depth": 6, "learning_rate": 0.025, "l2_leaf_reg": 12.0},
    {"iterations": 900,  "depth": 6, "learning_rate": 0.05,  "l2_leaf_reg": 6.0},
    {"iterations": 1200, "depth": 6, "learning_rate": 0.01,  "l2_leaf_reg": 5.0},
    {"iterations": 1200, "depth": 7, "learning_rate": 0.02,  "l2_leaf_reg": 5.0},
    {"iterations": 1200, "depth": 7, "learning_rate": 0.03,  "l2_leaf_reg": 10.0},
    {"iterations": 1500, "depth": 8, "learning_rate": 0.01,  "l2_leaf_reg": 3.0},
    {"iterations": 1500, "depth": 8, "learning_rate": 0.015, "l2_leaf_reg": 8.0},
    {"iterations": 900,  "depth": 6, "learning_rate": 0.025, "l2_leaf_reg": 12.0, "bagging_temperature": 0.5},
    {"iterations": 1200, "depth": 7, "learning_rate": 0.02,  "l2_leaf_reg": 5.0,  "bagging_temperature": 0.3},
]

search_rows = []
best = None

for i, params in enumerate(param_grid, start=1):
    model = CatBoostClassifier(**base_params, **params)
    model.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True,
        early_stopping_rounds=80,
    )

    val_proba = model.predict_proba(val_pool)[:, 1]
    threshold = utils.find_best_threshold_f2(y_val, val_proba)
    val_metrics = utils.compute_metrics(y_val, val_proba, threshold)

    row = {
        "candidate": i,
        **params,
        "best_iteration": int(model.get_best_iteration() or params["iterations"]),
        "threshold": threshold,
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    search_rows.append(row)

    score = (row["val_f2"], row["val_roc_auc"])
    if best is None or score > best["score"]:
        best = {"score": score, "model": model, "params": params, "threshold": threshold}

search_df = pd.DataFrame(search_rows).sort_values(
    ["val_f2", "val_roc_auc"], ascending=False
)
search_df

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


,candidate,iterations,depth,learning_rate,l2_leaf_reg,best_iteration,threshold,val_roc_auc,val_precision,val_recall,val_f2,val_tp,val_fp,val_fn,val_tn,bagging_temperature
4,5,900,6,0.025,12.0,595,0.422130,0.647965,0.118165,0.748408,0.362123,940,7015,316,5723,NaN
12,13,1200,7,0.020,5.0,1094,0.425756,0.650015,0.119775,0.730096,0.361593,917,6739,339,5999,0.3
3,4,900,5,0.050,10.0,317,0.398672,0.646047,0.110026,0.843153,0.361458,1059,8566,197,4172,NaN
7,8,1200,7,0.020,5.0,451,0.413657,0.645707,0.113709,0.793790,0.361441,997,7771,259,4967,NaN
11,12,900,6,0.025,12.0,755,0.416879,0.648245,0.116327,0.759554,0.360681,954,7247,302,5491,0.5
9,10,1500,8,0.010,3.0,67,0.450974,0.640751,0.110639,0.828822,0.360632,1041,8368,215,4370,NaN
0,1,600,4,0.050,3.0,128,0.430719,0.640787,0.118494,0.734076,0.360016,922,6859,334,5879,NaN
2,3,900,5,0.030,3.0,193,0.429023,0.642390,0.117528,0.740446,0.359434,930,6983,326,5755,NaN
10,11,1500,8,0.015,8.0,515,0.393375,0.646340,0.108709,0.848726,0.359407,1066,8740,190,3998,NaN
6,7,1200,6,0.010,5.0,48,0.463630,0.639048,0.117409,0.738854,0.358911,928,6976,328,5762,NaN


## 4. Метрики лучшей модели

Беру лучшую модель из шага 3 (она уже после `use_best_model` хранит оптимальную итерацию) и получаю `predict_proba` для train, val и test.

Здесь метрики считаются при двух разных порогах - и это не лишнее. Первый порог - F2 (≈ 0.416), тот же, что нашёлся в шаге 3. Это основной протокол всего проекта. На test recall выходит около 0.785: модель находит ~79% реальных случаев ранней реадмиссии. Precision при этом 0.113 - из 100 пациентов, которых модель пометила как риск, реально вернутся примерно 11. Ложных тревог много: FP = 7754. Это та плата, которую мы согласились нести ради высокого recall.



Сравнение с LogReg. CatBoost на test: ROC-AUC ≈ 0.650, F2 ≈ 0.358, recall ≈ 0.785. LogReg в ноутбуке 03: ROC-AUC ≈ 0.653, F2 ≈ 0.359, recall ≈ 0.751. По двум первым метрикам CatBoost практически не отличается от линейной модели - разница в шуме. Заметно лучше он только по recall. Из этого напрашивается вывод: сигнал в данных в основном линейный, и нелинейность бустинга мало что добавляет. Финальный ответ "бустинг или нейросеть лучше" станет понятен после TabM.

Разрыв train vs test: ROC-AUC на train ≈ 0.670, на test ≈ 0.650 - около 0.02. У LogReg был меньше (0.013), но и здесь не критично. Early stopping и L2 не дали разъехаться сильнее.

В словарь `metrics` собираю всё, что понадобится дальше: лучшие гиперпараметры, оба порога, метрики на трёх сплитах при каждом пороге, и полную таблицу всех 13 кандидатов.

In [5]:
best_model = best["model"]
best_threshold = best["threshold"]

train_proba = best_model.predict_proba(train_pool)[:, 1]
val_proba   = best_model.predict_proba(val_pool)[:, 1]
test_proba  = best_model.predict_proba(test_pool)[:, 1]

# F2-optimal threshold (main protocol — recall-weighted)
threshold_f2 = best_threshold


metrics = {
    "model": "catboost",
    "selection_metric": "validation F2 with validation-optimized threshold",
    "best_params": best["params"],
    "best_iteration": int(best_model.get_best_iteration() or best["params"]["iterations"]),
    "threshold_f2": threshold_f2,
    # Metrics at F2 threshold (recall-oriented, original protocol)
    "train": utils.compute_metrics(y_train, train_proba, threshold_f2),
    "val":   utils.compute_metrics(y_val,   val_proba,   threshold_f2),
    "test":  utils.compute_metrics(y_test,  test_proba,  threshold_f2),
    "hyperparameter_search": search_rows,
}

print(f"F2-threshold: {threshold_f2:.4f}  →  test precision={metrics['test']['precision']:.3f}  recall={metrics['test']['recall']:.3f}  FP={metrics['test']['fp']}")

pd.DataFrame(
    [metrics[split] for split in ["train", "val", "test"]],
    index=["train", "val", "test"]
)

F2-threshold: 0.4221  →  test precision=0.117  recall=0.750  FP=7094


,roc_auc,precision,recall,f2,tp,fp,fn,tn
train,0.701693,0.125614,0.801381,0.386032,3018,21008,748,17208
val,0.647965,0.118165,0.748408,0.362123,940,7015,316,5723
test,0.658565,0.117113,0.749801,0.360398,941,7094,314,5645


## 5. Сохранение артефактов

Готовлю файлы под ноутбук 08. Саму обученную модель не сохраняю - это тяжёлый файл, а переобучить одну модель не такая большая проблема. Сохраняю то, ради чего она работала, - предсказания и метрики.

В `results/predictions/catboost_val.csv` и `catboost_test.csv` для каждой строки лежат три числа: настоящая метка `y_true`, предсказанная вероятность `y_proba` и бинарный ответ `y_pred` при F2-пороге. У LogReg в той же папке лежат точно такие же файлы.

В `results/metrics/catboost.json` - сводный отчёт: лучшие параметры, оба порога, метрики на train/val/test при каждом пороге и полная история перебора. Дальше все ноутбуки берут результаты CatBoost именно отсюда и ничего не пересчитывают.

Сам ноутбук 08 (`08_comparison.ipynb`) модели не переобучает. Он читает уже сохранённые CSV и JSON каждой модели и из них собирает общее сравнение: совмещённый ROC, таблицы метрик, корреляции предсказаний между моделями. Формат файлов одинаковый для всех пяти моделей именно ради этого - чтобы 08-й работал универсально и ему не надо было ничего знать про внутренности каждой модели.

In [6]:
def save_predictions(path: Path, y_true: np.ndarray, y_proba: np.ndarray, threshold: float) -> None:
    pd.DataFrame(
        {
            "y_true": y_true,
            "y_proba": y_proba,
            "y_pred": (y_proba >= threshold).astype(int),
        }
    ).to_csv(path, index=False)


save_predictions(PREDICTIONS_DIR / "catboost_val.csv", y_val, val_proba, best_threshold)
save_predictions(PREDICTIONS_DIR / "catboost_test.csv", y_test, test_proba, best_threshold)

with open(METRICS_DIR / "catboost.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("saved:", METRICS_DIR / "catboost.json")
print("saved:", PREDICTIONS_DIR / "catboost_val.csv")
print("saved:", PREDICTIONS_DIR / "catboost_test.csv")

saved: ..\results\metrics\catboost.json
saved: ..\results\predictions\catboost_val.csv
saved: ..\results\predictions\catboost_test.csv


## Вывод

CatBoost обучен на тех же `train/val/test`, что и LogReg в ноутбуке 03 - одни данные, одни сплиты. Для сравнения моделей это критично. Категориальные признаки переданы нативно через `cat_features`, без OneHot. Гиперпараметры и порог отсечения подбирались только на val. Test использован один раз, на финале.

По цифрам CatBoost на ROC-AUC и F2 практически не отличается от логистической регрессии - разница в пределах шума. По recall он заметно выше (~0.78 против ~0.75). Это можно интерпретировать так: основная сигнальная часть в данных линейная, и нелинейность бустинга даёт лишь небольшой бонус. Для нас это полезный контекст - значит, у нейросетевых моделей будет такой же потолок, и оценивать TabM нужно прежде всего по тому, сможет ли она его пробить.

Артефакты (`catboost.json`, `catboost_val.csv`, `catboost_test.csv`) уже лежат в `results/`. Дальше их подхватит `08_comparison.ipynb`, и CatBoost станет одной из пяти моделей в общем сравнении вместе с LogReg, MLP, Transformer и TabM.